# Phần 2.2 – Bộ lọc thông thấp và thông cao

Notebook xử lý các ảnh trong `images/`, áp dụng bộ lọc Mean, Gaussian, Laplacian, Sobel và kernel custom, sau đó lưu ảnh so sánh cùng các metric vào `results/`.

Chạy các ô theo thứ tự từ trên xuống dưới. Mỗi lần chạy, notebook dọn các kết quả do chính nó tạo ở lần trước rồi sinh lại kết quả cho các ảnh đầu vào hiện có.


## 1. Thiết lập và cấu hình

Thư viện được khai báo trong `requirements.txt`. Chương trình tự tìm thư mục `part2_filters` từ thư mục làm việc của Jupyter. Ảnh đầu vào hỗ trợ PNG, JPG, JPEG và BMP.


In [ ]:

from pathlib import Path
import csv

import cv2
import matplotlib.pyplot as plt
import numpy as np


# Lay duong dan dua tren vi tri file code, nen co the chay tu moi thu muc.
# Find this notebook's project directory from the Jupyter kernel working directory.
CWD = Path.cwd().resolve()
ROOT = None
for base in (CWD, *CWD.parents):
    if (base / "images").is_dir() and (base / "filter_part2.ipynb").is_file():
        ROOT = base
        break
    candidate = base / "part2_filters"
    if (candidate / "images").is_dir() and (candidate / "filter_part2.ipynb").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError(
        "Could not locate part2_filters. Start Jupyter from the project folder "
        "or from part2_filters/."
    )
INPUT_DIR = ROOT / "images"
OUTPUT_DIR = ROOT / "results"

# Cac dinh dang anh ma chuong trinh se tu dong xu ly trong images/.
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}
CUSTOM_IMAGE_NAME = "urban.png"

# Kernel custom: lay thong tin tu 8 pixel xung quanh, khong giu truc tiep pixel tam.
# Trong so lon hon o 4 huong ngang/doc; tong he so bang 1.
CUSTOM_KERNEL = np.array(
    [[1, 2, 1], [2, 0, 2], [1, 2, 1]], dtype=np.float32
) / 12

# Chi xoa cac file co dung hau to do chuong trinh nay tao ra.
GENERATED_SUFFIXES = (
    "_mean.png",
    "_gaussian.png",
    "_laplacian.png",
    "_sobel.png",
    "_comparison.png",
    "_custom.png",
    "_custom_comparison.png",
    "metrics.csv",
)




## 2. Hàm metric và dọn kết quả cũ

MSE đo mức thay đổi so với ảnh gốc sau khi chuẩn hóa pixel về `[0, 1]`. Edge energy là trung bình độ lớn gradient Sobel trên ảnh xám.


In [ ]:
def clear_old_results():
    """Xoa ket qua cua lan chay truoc de output chi con anh hien tai."""
    removed = 0
    for path in OUTPUT_DIR.iterdir():
        if path.is_file() and path.name.endswith(GENERATED_SUFFIXES):
            path.unlink()
            removed += 1
    return removed


def normalized_mse(original, filtered):
    """MSE tren anh goc va anh loc, sau khi dua pixel ve [0, 1]."""
    original = original.astype(np.float32) / 255.0
    filtered = filtered.astype(np.float32) / 255.0
    return float(np.mean((original - filtered) ** 2))


def edge_energy(gray_image):
    """Trung binh do lon gradient Sobel; gia tri thap hon nghia la it bien hon."""
    gx = cv2.Sobel(gray_image, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_image, cv2.CV_32F, 0, 1, ksize=3)
    return float(cv2.magnitude(gx, gy).mean())


## 3. Xử lý từng ảnh

Mỗi ảnh được làm trơn bằng Mean và Gaussian, đồng thời tạo bản đồ biên Laplacian và Sobel. Ảnh `urban.png` được xử lý thêm bằng kernel custom.


In [ ]:
def process_image(input_path):
    """Doc mot anh, loc anh va luu cac ket qua."""
    image = cv2.imread(str(input_path))

    if image is None:
        print(f"Khong doc duoc: {input_path}")
        return []

    # 1. Chuyen sang anh xam de thuc hien high-pass.
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 2. Low-pass: lam tron anh va giam chi tiet/nhieu nho.
    mean = cv2.blur(image, (5, 5))
    gaussian = cv2.GaussianBlur(image, (5, 5), 0)

    # 3. High-pass: lam noi bien va chi tiet.
    laplacian = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    laplacian = cv2.convertScaleAbs(laplacian)

    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel = cv2.magnitude(sobel_x.astype("float32"), sobel_y.astype("float32"))
    sobel = cv2.convertScaleAbs(sobel)

    # 4. Luu tung anh de tien chen vao bao cao.
    name = input_path.stem
    cv2.imwrite(str(OUTPUT_DIR / f"{name}_mean.png"), mean)
    cv2.imwrite(str(OUTPUT_DIR / f"{name}_gaussian.png"), gaussian)
    cv2.imwrite(str(OUTPUT_DIR / f"{name}_laplacian.png"), laplacian)
    cv2.imwrite(str(OUTPUT_DIR / f"{name}_sobel.png"), sobel)

    # Bo loc custom chi duoc ap dung len urban.png.
    if input_path.name.lower() == CUSTOM_IMAGE_NAME.lower():
        custom = cv2.filter2D(
            image, -1, CUSTOM_KERNEL, borderType=cv2.BORDER_REFLECT
        )
        cv2.imwrite(str(OUTPUT_DIR / f"{name}_custom.png"), custom)

    # 5. Tao mot hinh tong hop de so sanh truc quan.
    images = [
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(mean, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(gaussian, cv2.COLOR_BGR2RGB),
        gray,
        laplacian,
        sobel,
    ]
    titles = [
        "Original",
        "Mean 5x5",
        "Gaussian 5x5",
        "Grayscale",
        "Laplacian",
        "Sobel",
    ]

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, result, title in zip(axes.flat, images, titles):
        if result.ndim == 2:
            ax.imshow(result, cmap="gray")
        else:
            ax.imshow(result)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name}_comparison.png", dpi=150)
    plt.close()

    if input_path.name.lower() == CUSTOM_IMAGE_NAME.lower():
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original")
        axes[1].imshow(cv2.cvtColor(custom, cv2.COLOR_BGR2RGB))
        axes[1].set_title("Custom 3x3")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"{name}_custom_comparison.png", dpi=150)
        plt.close()

    # Metric: MSE do muc thay doi va edge_energy do luong bien con lai.
    metric_images = {
        "original": gray,
        "mean": cv2.cvtColor(mean, cv2.COLOR_BGR2GRAY),
        "gaussian": cv2.cvtColor(gaussian, cv2.COLOR_BGR2GRAY),
    }
    if input_path.name.lower() == CUSTOM_IMAGE_NAME.lower():
        metric_images["custom"] = cv2.cvtColor(custom, cv2.COLOR_BGR2GRAY)

    metrics = []
    for method, filtered_gray in metric_images.items():
        metrics.append({
            "image": name,
            "method": method,
            "mse_from_original": round(normalized_mse(gray, filtered_gray), 6),
            "edge_energy": round(
                edge_energy(filtered_gray.astype(np.float32) / 255.0), 6
            ),
        })

    print(f"Da xu ly: {input_path.name}")
    return metrics


## 4. Chạy toàn bộ bộ ảnh

Ô cuối bên dưới chạy pipeline cho tất cả ảnh hợp lệ trong `images/` và ghi `metrics.csv` cùng các ảnh kết quả vào `results/`.


In [ ]:
def main():
    OUTPUT_DIR.mkdir(exist_ok=True)
    removed = clear_old_results()

    # Tu dong lay tat ca anh trong images/, khong phu thuoc vao so luong anh.
    image_paths = sorted(
        path for path in INPUT_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        and path.name != "source_contact_sheet.png"
    )

    if not image_paths:
        print("Khong tim thay anh PNG/JPG/JPEG/BMP trong thu muc images/.")
        return

    if not any(path.name.lower() == CUSTOM_IMAGE_NAME.lower() for path in image_paths):
        print(f"Khong tim thay {CUSTOM_IMAGE_NAME}; bo qua bo loc custom.")

    all_metrics = []
    for input_path in image_paths:
        all_metrics.extend(process_image(input_path))

    with (OUTPUT_DIR / "metrics.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file, fieldnames=["image", "method", "mse_from_original", "edge_energy"]
        )
        writer.writeheader()
        writer.writerows(all_metrics)

    print(f"\nDa xoa {removed} anh ket qua cu.")
    print(f"Da xu ly {len(image_paths)} anh.")
    print(f"Tat ca ket qua duoc luu tai: {OUTPUT_DIR}")


main()
